# 📓 Semana 14 · Dia 3 — Text-to-SQL em produção: seguro e confiável

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Agente SQL seguro e auditável |

---


## 📖 Teoria — Text-to-SQL é o caso de uso nº 1 de dados

Transformar pergunta em SQL é o que o mercado mais pede. Os 4 pilares:

1. **Dicionário de dados** no prompt (tabelas, colunas, exemplos)
2. **Segurança**: bloquear DELETE/UPDATE/DROP/INSERT — só leitura
3. **Auto-correção**: se o SQL falhar, o agente corrige (1–2 tentativas)
4. **RLS embutida**: o usuário só vê os dados que tem permissão


### 💻 Na prática — Dicionário de dados

Monte o dicionário das tabelas Ouro para o prompt.


In [ ]:
# Dicionário de dados (versionado!)
dicionario = """
workspace.ouro.vendas_por_dia: data_venda (DATE), receita_total (DOUBLE), n_vendas, n_notas
workspace.ouro.receita_por_pais: Country (STRING), receita_total (DOUBLE), n_vendas
workspace.ouro.top_produtos: Description (STRING), StockCode, receita_total
Regra: receita = Quantity * UnitPrice. Datas no formato YYYY-MM-DD.
"""
print(dicionario)

### 💻 Na prática — Segurança anti-DML

Valide o SQL gerado antes de executar.


In [ ]:
# Validador: só permite SELECT
import re
def valida_sql(sql):
    proibido = re.findall(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|TRUNCATE|CREATE)\b",
                         sql, re.IGNORECASE)
    if proibido:
        raise ValueError(f"SQL bloqueado (operação proibida: {proibido})")
    if not sql.strip().upper().startswith("SELECT"):
        raise ValueError("Apenas SELECT é permitido")
    return sql
print(valida_sql("SELECT * FROM workspace.ouro.vendas_por_dia"))
try:
    valida_sql("DELETE FROM workspace.ouro.vendas_por_dia")
except ValueError as e:
    print("Bloqueado:", e)

### 💻 Na prática — Agente com auto-correção

Gere SQL → valide → execute → corrija se falhar.


In [ ]:
# Fluxo Text-to-SQL com auto-correção
def perguntar_sql(pergunta, max_tentativas=2):
    for tentativa in range(max_tentativas):
        sql_gerado = llm.invoke(dicionario + "\nPergunta: " + pergunta
                                + "\nResponda apenas com o SQL.").content
        sql_gerado = sql_gerado.strip().strip("```sql").strip("```").strip()
        try:
            valida_sql(sql_gerado)
            return spark.sql(sql_gerado).toPandas()
        except Exception as e:
            if tentativa == max_tentativas - 1:
                return f"Falhou após {max_tentativas} tentativas: {e}"
            print(f"Tentativa {tentativa+1} falhou ({e}); corrigindo...")
print("Função de Text-to-SQL com validação e auto-correção pronta.")

> 🎯 **Dica de prova**: Agentes/entrevista: Text-to-SQL exige dicionário de dados, whitelist de operação (SELECT only), RLS e auto-correção limitada. Pergunta: 'como evitar que o agente apague dados?' → validação de SQL.


## 🎯 Exercícios de fixação

**1.** Adicione limite de linhas (LIMIT 100) automático ao SQL gerado.

**2.** Por que limitar as tentativas de auto-correção?

**3.** Como a RLS (Semana 7) protege o Text-to-SQL?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** LIMIT

Acrescente `LIMIT 100` se não houver — evita queries gigantes e custo.

**2.** Tentativas

Custo e loops infinitos — 1–2 correções bastam; depois responda com erro.

**3.** RLS

O agente roda com as permissões do usuário (dynamic views) — mesmo que gere SQL livre, só vê o que o usuário pode ver.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*